# Simulador base

Reencena os pedidos históricos da Olist (2016–2018) como se acontecessem a partir de 2026-09-06 e grava um lote diário por entidade em `olist.landing.raw_data/<entidade>/sim_date=AAAA-MM-DD/`.

**Parâmetros**

| Widget | Padrão | Uso |
|---|---|---|
| `dias_por_execucao` | `1` | quantos dias simulados esta execução gera (1 a 30) |
| `sim_date_override` | vazio | reprocessa um dia específico (`AAAA-MM-DD`); vazio = continua de onde parou |
| `run_id` | vazio | identificador da execução; no job recebe `{{job.run_id}}`, vazio gera um UUID |

O progresso fica em `olist.ops.simulator_runs`. Desenho: `docs/design/fase-1-simulador-bronze.md`.

In [0]:
# O QUE FAZ: cria os 3 parâmetros do notebook. Eles aparecem como caixas de texto no topo da página.
#   Na execução interativa, você preenche à mão; no job, os valores vêm dos parâmetros da task.
#
# FUNÇÕES:
# - dbutils.widgets.text(nome, valor_padrão, rótulo): cria um widget de texto. "nome" é usado no
#   código para ler o valor; "rótulo" é o texto exibido na tela. Se o widget já existe, nada muda.
#   O valor é lido depois com dbutils.widgets.get(nome) e sempre chega como string.
dbutils.widgets.text("dias_por_execucao", "1", "Dias por execução")
dbutils.widgets.text("sim_date_override", "", "Reprocessar dia (AAAA-MM-DD)")
dbutils.widgets.text("run_id", "", "ID da execução")

In [0]:
# O QUE FAZ: define as constantes do simulador: caminhos dos Volumes, tabela de controle e regras
#   de datas. Todos os caminhos derivam de CATALOG: trocar o catálogo é mudar uma linha só.
#
# RECURSOS:
# - datetime.date(ano, mês, dia): data de calendário sem horário (tipo nativo do Python).
# - f"...{variavel}...": f-string, monta texto substituindo as variáveis entre chaves.
# - /Volumes/<catálogo>/<schema>/<volume>: caminho de arquivos de um Volume do Unity Catalog,
#   a forma de guardar arquivos no serverless.
from datetime import date

CATALOG = "olist"

SOURCE_DIR = f"/Volumes/{CATALOG}/landing/olist_source"
RAW_DIR = f"/Volumes/{CATALOG}/landing/raw_data"
STAGING_DIR = f"{RAW_DIR}/_staging"  # prefixo "_": o Auto Loader ignora a pasta

RUNS_TABLE = f"{CATALOG}.ops.simulator_runs"

DATE_OFFSET_DAYS = 3654          # 522 semanas: preserva o dia da semana
SIM_START = date(2026, 9, 6)     # primeiro pedido (2016-09-04) + 3654 dias
MAX_DIAS_POR_EXECUCAO = 30

In [0]:
# O QUE FAZ: lê os 3 widgets, converte para os tipos certos (int, date, str) e valida.
#   Um valor inválido para a execução aqui, antes de qualquer leitura ou escrita.
#
# FUNÇÃO DECLARADA:
# - parse_params(dias_raw, override_raw, run_id_raw): recebe os textos dos widgets e devolve
#   (dias, override, run_id). É pura: não usa Spark nem dbutils, então pode ser testada com pytest.
#   Regras: dias inteiro de 1 a 30; override vazio ou data >= SIM_START; run_id vazio vira UUID.
#
# FUNÇÕES NATIVAS:
# - uuid.uuid4(): gera um identificador aleatório único (ex.: "3f2b...-..."); str() vira texto.
# - str.strip(): remove espaços e quebras de linha do início e do fim do texto.
# - int(texto): converte texto em inteiro; lança ValueError se não for número.
# - date.fromisoformat("AAAA-MM-DD"): converte texto em data; lança ValueError se o formato estiver errado.
# - raise ... from None: lança o novo erro escondendo o erro original, para a mensagem ficar limpa.
# - {valor!r}: mostra o valor com repr(), com aspas, o que deixa visível um texto vazio ou com espaços.
# - "a or b": devolve "a" se ele tiver valor (texto não vazio); senão, devolve "b".
# - dbutils.widgets.get(nome): lê o valor atual do widget (sempre string).
import uuid


def parse_params(dias_raw: str, override_raw: str, run_id_raw: str) -> tuple[int, date | None, str]:
    """Converte e valida os valores dos widgets. Não usa Spark nem dbutils: pode ser testada isoladamente."""
    try:
        dias = int(dias_raw.strip())
    except ValueError:
        raise ValueError(f"dias_por_execucao deve ser um inteiro; recebido: {dias_raw!r}") from None
    if not 1 <= dias <= MAX_DIAS_POR_EXECUCAO:
        raise ValueError(f"dias_por_execucao deve estar entre 1 e {MAX_DIAS_POR_EXECUCAO}; recebido: {dias}")

    override = None
    if override_raw.strip():
        try:
            override = date.fromisoformat(override_raw.strip())
        except ValueError:
            raise ValueError(f"sim_date_override deve estar no formato AAAA-MM-DD; recebido: {override_raw!r}") from None
        if override < SIM_START:
            raise ValueError(f"sim_date_override não pode ser anterior a {SIM_START}; recebido: {override}")

    run_id = run_id_raw.strip() or str(uuid.uuid4())
    return dias, override, run_id


DIAS_POR_EXECUCAO, SIM_DATE_OVERRIDE, RUN_ID = parse_params(
    dbutils.widgets.get("dias_por_execucao"),
    dbutils.widgets.get("sim_date_override"),
    dbutils.widgets.get("run_id"),
)

print(f"run_id:            {RUN_ID}")
print(f"dias_por_execucao: {DIAS_POR_EXECUCAO}")
print(f"sim_date_override: {SIM_DATE_OVERRIDE or '(nenhum: continua de onde parou)'}")

## Leitura da fonte e deslocamento de datas

Lê os 9 CSVs originais de `olist_source` com todas as colunas como texto e desloca as colunas de data em `DATE_OFFSET_DAYS` dias. Nada é gravado nesta etapa: as transformações do Spark são *lazy* e só executam quando um resultado é pedido.

In [0]:
# O QUE FAZ: mapeia cada entidade da landing para o CSV de origem e lista as colunas de data/hora
#   que precisam ser deslocadas em 3.654 dias. Só constantes: nenhum dado é lido aqui.
#
# RECURSOS:
# - dict {chave: valor}: dicionário; SOURCE_FILES["orders"] devolve o nome do arquivo.
# - TS_FORMAT: padrão de data/hora do Spark ("yyyy" ano, "MM" mês, "dd" dia, "HH" hora 0-23,
#   "mm" minuto, "ss" segundo). Atenção: "MM" é mês e "mm" é minuto.
# Chaves = nomes das entidades na landing (desenho A2)
SOURCE_FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"

# Colunas de data/hora por entidade; as demais entidades não têm datas
DATE_COLUMNS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

In [0]:
# O QUE FAZ: declara as duas funções de leitura: ler o CSV de uma entidade e deslocar as datas.
#
# FUNÇÕES DECLARADAS:
# - read_source(entity): devolve um DataFrame com o CSV original da entidade, tudo como string.
# - shift_dates(df, columns, days): soma "days" dias às colunas de data indicadas e devolve o
#   resultado de novo como texto. Se a lista de colunas estiver vazia, devolve o df sem mudança.
#
# FUNÇÕES NATIVAS:
# - spark.read.option(chave, valor).csv(caminho): leitor de CSV. Opções usadas:
#     header=True     -> a primeira linha tem os nomes das colunas;
#     multiLine=True  -> permite campo entre aspas com quebra de linha (comentários das avaliações);
#     escape='"'      -> aspas dentro de um campo vêm duplicadas (""), padrão do CSV da Olist.
#   Sem inferSchema, todas as colunas são string.
# - df.withColumns({nome: expressão}): cria ou substitui várias colunas de uma vez (uma projeção
#   só, mais eficiente que encadear vários withColumn).
# - F.col(nome): referência a uma coluna. F.lit(valor): valor constante como coluna.
# - F.to_timestamp_ntz(coluna, formato): texto -> TIMESTAMP_NTZ (data/hora SEM fuso). Sem fuso,
#   o horário de verão e o fuso da sessão não alteram o horário ao somar dias.
# - F.make_dt_interval(days=n): cria um intervalo de n dias; "timestamp + intervalo" soma os dias.
# - F.date_format(timestamp, formato): data/hora -> texto no formato indicado.
# - {c: ... for c in columns}: dict comprehension, monta o dicionário com uma expressão por coluna.
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def read_source(entity: str) -> DataFrame:
    """Lê o CSV original da entidade. Sem inferSchema: todas as colunas chegam como string."""
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)  # comentários das avaliações têm quebra de linha
        .option("escape", '"')
        .csv(f"{SOURCE_DIR}/{SOURCE_FILES[entity]}")
    )


def shift_dates(df: DataFrame, columns: list[str], days: int) -> DataFrame:
    """Desloca colunas de data/hora em `days` dias. Entrada e saída em texto no formato TS_FORMAT; nulos continuam nulos."""
    if not columns:
        return df
    return df.withColumns({
        c: F.date_format(
            F.to_timestamp_ntz(F.col(c), F.lit(TS_FORMAT)) + F.make_dt_interval(days=F.lit(days)),
            TS_FORMAT,
        )
        for c in columns
    })

In [0]:
# O QUE FAZ: monta o DataFrame de cada entidade de origem, já com as datas deslocadas.
#   É lazy (preguiçoso): o Spark só registra o plano. Lê apenas a primeira linha de cada CSV para
#   descobrir os nomes das colunas; os dados são lidos quando uma ação (count, write...) roda.
#
# RECURSOS:
# - DATE_COLUMNS.get(entity, []): busca no dicionário; se a entidade não tem datas, devolve [].
# - ", ".join(lista): junta os textos da lista separados por vírgula.
sources = {
    entity: shift_dates(read_source(entity), DATE_COLUMNS.get(entity, []), DATE_OFFSET_DAYS)
    for entity in SOURCE_FILES
}

print(f"{len(sources)} entidades carregadas: {', '.join(sources)}")

In [0]:
# O QUE FAZ: valida o deslocamento de datas (critério de aceite nº 5): todo pedido deve ter
#   andado exatamente 3.654 dias, no mesmo dia da semana, de 2026-09-06 a 2028-10-18.
#
# FUNÇÕES NATIVAS:
# - df.select(...): escolhe ou calcula colunas. .alias(nome): renomeia a coluna resultante.
# - df.transform(funcao, *args): chama funcao(df, *args); permite encadear funções próprias.
# - F.to_date(coluna): texto ou timestamp -> DATE (descarta o horário).
# - df.agg(...): agrega a tabela inteira em uma linha.
# - F.datediff(fim, inicio): diferença em dias entre duas datas.
# - F.min / F.max: menor e maior valor.
# - F.count_if(condição): conta as linhas em que a condição é verdadeira.
# - F.dayofweek(data): dia da semana (1 = domingo ... 7 = sábado).
# - display(df): ação do Databricks; executa o plano e mostra o resultado em tabela.
# - df.limit(n): só as n primeiras linhas.
validacao = (
    read_source("orders")
    .select(F.col("order_purchase_timestamp").alias("original"), "order_purchase_timestamp")
    .transform(shift_dates, ["order_purchase_timestamp"], DATE_OFFSET_DAYS)
    .select(
        F.to_date("original").alias("data_original"),
        F.to_date("order_purchase_timestamp").alias("data_simulada"),
    )
    .agg(
        F.min(F.datediff("data_simulada", "data_original")).alias("min_dias"),
        F.max(F.datediff("data_simulada", "data_original")).alias("max_dias"),
        F.count_if(F.dayofweek("data_simulada") != F.dayofweek("data_original")).alias("dia_semana_diferente"),
        F.min("data_simulada").alias("primeira_compra"),
        F.max("data_simulada").alias("ultima_compra"),
    )
)

display(validacao)
display(sources["orders"].limit(5))

## Montagem das entidades

Transforma as 9 tabelas de origem nas 10 entidades da landing (desenho A2). Cada entidade recebe a coluna `sim_date`, o dia simulado em que o registro é emitido:

| Entidade | `sim_date` |
|---|---|
| `orders` (cabeçalho) | dia da compra |
| `order_status_events` | dia de cada evento (`created`, `approved`, `shipped`, `delivered` e status final) |
| `order_items`, `order_payments` | dia da compra do pedido |
| `order_reviews` | dia da resposta da avaliação |
| `customers` (CDC) | dia do primeiro pedido do cliente |
| `sellers` (CDC), `products`, `geolocation`, `category_translation` | carga inicial no primeiro dia (`SIM_START`) |

Registros com `sim_date` posterior ao último dia do dataset nunca são emitidos, porque a execução só gera dias até esse limite (Parte 4).

In [0]:
# O QUE FAZ: constantes da montagem das entidades: início da simulação como texto de data/hora,
#   status finais que viram evento extra e as colunas do cabeçalho do pedido (desenho A2).
#
# RECURSOS:
# - list [...]: lista ordenada. Inteiros como 5 são constantes nomeadas para não espalhar
#   "números mágicos" pelo código.
SIM_START_TS = f"{SIM_START} 00:00:00"

# Status finais que não têm coluna de data própria: viram um evento extra (sequência 5)
FINAL_STATUSES = ["canceled", "unavailable", "invoiced", "processing"]
FINAL_EVENT_SEQUENCE = 5

ORDER_HEADER_COLUMNS = [
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]

In [0]:
# O QUE FAZ: declara as funções puras (DataFrame -> DataFrame) que transformam as 9 tabelas de
#   origem nas 10 entidades da landing, cada uma com a coluna sim_date (dia em que é emitida).
#
# FUNÇÕES DECLARADAS:
# - with_sim_date(df, ts_column): cria sim_date com a data (sem horário) da coluna indicada.
# - with_initial_load(df): sim_date = SIM_START para todas as linhas (carga inicial no 1º dia).
# - with_cdc_insert(df, change_ts): acrescenta as colunas de CDC do registro de inclusão:
#   operation = "INSERT", change_ts = momento da mudança, change_sequence = 1.
# - build_orders(orders): cabeçalho do pedido, só as colunas conhecidas na compra; dia = compra.
# - build_status_events(orders): transforma as datas do pedido em eventos (uma linha por status):
#   created(1), approved(2), shipped(3), delivered(4); data nula = etapa que não aconteceu, sem
#   evento. Pedidos com status final (canceled etc.) ganham um evento 5 na última data conhecida.
# - build_order_children(df, orders): itens/pagamentos recebem o sim_date da compra do pedido.
# - build_customers(customers, orders): cliente emitido como CDC INSERT no dia do seu pedido.
# - build_entities(sources): aplica as funções acima e devolve o dicionário das 10 entidades.
#
# FUNÇÕES NATIVAS:
# - Column: tipo de uma expressão de coluna do Spark (usado só na anotação de tipo).
# - df.withColumn(nome, expressão): cria ou substitui uma coluna.
# - F.stack(n, v1, v2, ...): gera n linhas a partir de cada linha de entrada, distribuindo os
#   valores em grupos (aqui 4 grupos de 3 valores: status, sequência, data). É um "unpivot".
#   .alias("a", "b", "c") dá nome às 3 colunas geradas.
# - df.where(condição): filtra linhas. Column.isNotNull(): verdadeiro se não for nulo.
# - Column.isin(lista): verdadeiro se o valor estiver na lista.
# - F.greatest(c1, c2, ...): o maior valor entre as colunas, ignorando nulos (nulo só se todas
#   forem nulas). Funciona em texto porque "yyyy-MM-dd HH:mm:ss" ordena como data.
# - df1.unionByName(df2): empilha as linhas casando as colunas pelo NOME (union casa pela posição).
# - df.join(outro, on=coluna, how="inner"): junta as tabelas pela coluna; inner mantém só as
#   linhas com correspondência nos dois lados.
# - df.drop(coluna): remove a coluna.
from pyspark.sql import Column


def with_sim_date(df: DataFrame, ts_column: str) -> DataFrame:
    """Define o dia simulado de emissão a partir de uma coluna de data/hora em texto."""
    return df.withColumn("sim_date", F.to_date(ts_column))


def with_initial_load(df: DataFrame) -> DataFrame:
    """Carga inicial: todos os registros emitidos no primeiro dia simulado."""
    return df.withColumn("sim_date", F.lit(SIM_START))


def with_cdc_insert(df: DataFrame, change_ts: Column) -> DataFrame:
    """Colunas de CDC para o registro de inclusão (desenho A2)."""
    return df.withColumns({
        "operation": F.lit("INSERT"),
        "change_ts": change_ts,
        "change_sequence": F.lit(1),
    })


def build_orders(orders: DataFrame) -> DataFrame:
    """Cabeçalho do pedido: só o que se conhece no momento da compra."""
    return with_sim_date(orders.select(*ORDER_HEADER_COLUMNS), "order_purchase_timestamp")


def build_status_events(orders: DataFrame) -> DataFrame:
    """Um registro por mudança de status. Datas nulas (etapa que não aconteceu) não geram evento."""
    timeline = (
        orders
        .select(
            "order_id",
            F.stack(
                F.lit(4),
                F.lit("created"),   F.lit(1), F.col("order_purchase_timestamp"),
                F.lit("approved"),  F.lit(2), F.col("order_approved_at"),
                F.lit("shipped"),   F.lit(3), F.col("order_delivered_carrier_date"),
                F.lit("delivered"), F.lit(4), F.col("order_delivered_customer_date"),
            ).alias("status", "event_sequence", "event_ts"),
        )
        .where(F.col("event_ts").isNotNull())
    )

    final_status = (
        orders
        .where(F.col("order_status").isin(FINAL_STATUSES))
        .select(
            "order_id",
            F.col("order_status").alias("status"),
            F.lit(FINAL_EVENT_SEQUENCE).alias("event_sequence"),
            # última data conhecida do pedido; greatest ignora nulos
            F.greatest(
                "order_purchase_timestamp",
                "order_approved_at",
                "order_delivered_carrier_date",
                "order_delivered_customer_date",
            ).alias("event_ts"),
        )
    )

    return with_sim_date(timeline.unionByName(final_status), "event_ts")


def build_order_children(df: DataFrame, orders: DataFrame) -> DataFrame:
    """Itens e pagamentos: emitidos no dia da compra do pedido a que pertencem."""
    purchase_dates = build_orders(orders).select("order_id", "sim_date")
    return df.join(purchase_dates, on="order_id", how="inner")


def build_customers(customers: DataFrame, orders: DataFrame) -> DataFrame:
    """Cliente nasce no checkout: inclusão (CDC INSERT) no dia do pedido."""
    first_order = orders.select("customer_id", "order_purchase_timestamp")
    return (
        customers
        .join(first_order, on="customer_id", how="inner")
        .transform(with_cdc_insert, F.col("order_purchase_timestamp"))
        .transform(with_sim_date, "order_purchase_timestamp")
        .drop("order_purchase_timestamp")
    )


def build_entities(sources: dict[str, DataFrame]) -> dict[str, DataFrame]:
    """Monta as 10 entidades da landing a partir das 9 tabelas de origem (datas já deslocadas)."""
    orders = sources["orders"]
    return {
        "orders": build_orders(orders),
        "order_status_events": build_status_events(orders),
        "order_items": build_order_children(sources["order_items"], orders),
        "order_payments": build_order_children(sources["order_payments"], orders),
        "order_reviews": with_sim_date(sources["order_reviews"], "review_answer_timestamp"),
        "customers": build_customers(sources["customers"], orders),
        "sellers": with_initial_load(with_cdc_insert(sources["sellers"], F.lit(SIM_START_TS))),
        "products": with_initial_load(sources["products"]),
        "geolocation": with_initial_load(sources["geolocation"]),
        "category_translation": with_initial_load(sources["category_translation"]),
    }

In [0]:
# O QUE FAZ: monta as 10 entidades. Continua lazy: nenhum dado é processado ainda.
entities = build_entities(sources)

print(f"{len(entities)} entidades montadas: {', '.join(entities)}")

In [0]:
# O QUE FAZ: confere se as contagens de linhas das entidades batem com as do setup (Fase 0).
#   order_status_events não tem valor esperado aqui (é derivada) e aparece com ok = true.
#
# FUNÇÕES NATIVAS:
# - df.count(): ação; executa o plano e devolve o número de linhas.
# - spark.createDataFrame(lista_de_tuplas, schema): cria um DataFrame pequeno a partir de dados
#   Python. O schema é uma string DDL: "nome tipo, nome tipo".
# - EXPECTED_ROWS.get(name): valor esperado, ou None se a entidade não estiver no dicionário.
# - Column.isNull(): verdadeiro se for nulo.
# - "|" entre colunas: OU lógico do Spark (cada lado entre parênteses, por precedência de operadores).
# - 99_441: o "_" é só separador visual de milhar em Python.
# Mesmas contagens validadas no setup; order_status_events é derivada e tem contagem própria
EXPECTED_ROWS = {
    "orders": 99_441,
    "order_items": 112_650,
    "order_payments": 103_886,
    "order_reviews": 99_224,
    "customers": 99_441,
    "sellers": 3_095,
    "products": 32_951,
    "geolocation": 1_000_163,
    "category_translation": 71,
}

contagens = spark.createDataFrame(
    [(name, df.count(), EXPECTED_ROWS.get(name)) for name, df in entities.items()],
    "entidade string, linhas long, esperado long",
).withColumn("ok", F.col("esperado").isNull() | (F.col("linhas") == F.col("esperado")))

display(contagens)

In [0]:
# O QUE FAZ: valida os eventos de status: quantidade por sequência/status e, para um pedido
#   entregue, os 4 eventos em dias diferentes (critério de aceite nº 6).
#
# FUNÇÕES NATIVAS:
# - df.groupBy(colunas).count(): agrupa e conta as linhas de cada grupo (coluna "count").
# - df.orderBy(colunas): ordena o resultado.
display(
    entities["order_status_events"]
    .groupBy("event_sequence", "status")
    .count()
    .orderBy("event_sequence", "status")
)

# Pedido entregue: 4 eventos, cada um no seu dia (critério de aceite nº 6)
display(
    entities["order_status_events"]
    .where(F.col("order_id") == "e481f51cbdc54678b7cc49136f2d6af7")
    .orderBy("event_sequence")
)

## Controle da execução

Define quais dias simulados esta execução gera e registra a execução como `RUNNING` em `olist.ops.simulator_runs` (desenho A1).

- **Sem `sim_date_override`:** continua do dia seguinte ao último `SUCCESS`; sem histórico, começa em `SIM_START`.
- **Com `sim_date_override`:** reprocessa a partir do dia informado, que pode ser qualquer dia já gerado ou o próximo dia, nunca um dia à frente (não pode haver lacunas).
- O último dia simulado (`SIM_END`) é o dia da última compra do dataset. Quando todos os dias já foram gerados, a execução termina sem fazer nada.

In [0]:
# O QUE FAZ: declara as funções que decidem QUAIS dias esta execução gera.
#
# FUNÇÕES DECLARADAS:
# - get_last_success_date(): consulta ops.simulator_runs e devolve o último dia gerado com
#   SUCCESS, ou None se o simulador nunca rodou (MAX de zero linhas = NULL).
# - compute_window(last_success, override, dias, sim_start, sim_end): função pura que devolve
#   (início, fim) da janela ou None se não há mais dias. Regras: o próximo dia é o seguinte ao
#   último sucesso; override só pode repetir um dia já gerado ou o próximo (sem pular dias);
#   o fim não passa de sim_end.
#
# FUNÇÕES NATIVAS:
# - timedelta(days=n): intervalo de n dias; data + timedelta = outra data.
# - spark.sql(texto): executa SQL e devolve um DataFrame.
# - df.first(): ação; devolve a primeira linha (Row). Row["coluna"] lê o valor da coluna.
# - min(a, b): o menor entre dois valores (função nativa do Python).
from datetime import timedelta


def get_last_success_date() -> date | None:
    """Último dia simulado gerado com sucesso; None se o simulador nunca rodou."""
    return spark.sql(
        f"SELECT MAX(sim_date_end) AS ultimo FROM {RUNS_TABLE} WHERE status = 'SUCCESS'"
    ).first()["ultimo"]


def compute_window(
    last_success: date | None,
    override: date | None,
    dias: int,
    sim_start: date,
    sim_end: date,
) -> tuple[date, date] | None:
    """Intervalo de dias [início, fim] desta execução; None se não há mais dias a gerar. Função pura."""
    next_date = sim_start if last_success is None else last_success + timedelta(days=1)

    if override is not None and override > next_date:
        raise ValueError(f"sim_date_override ({override}) pularia dias: o próximo dia a gerar é {next_date}")
    start = override or next_date

    if start > sim_end:
        return None
    end = min(start + timedelta(days=dias - 1), sim_end)
    return start, end

In [0]:
# O QUE FAZ: calcula o último dia da simulação (SIM_END = dia da última compra), a janela desta
#   execução, e encerra o notebook com sucesso se não houver mais dias a gerar.
#
# FUNÇÕES NATIVAS:
# - df.agg(F.max(...)).first()["fim"]: maior sim_date; first() é seguro aqui, pois o agg tem 1 linha.
# - dbutils.notebook.exit(texto): termina o notebook nesse ponto, com sucesso, devolvendo o texto
#   (no job, a task termina como bem-sucedida).
# - a, b = tupla: desempacota a tupla (início, fim) em duas variáveis.
SIM_END = entities["orders"].agg(F.max("sim_date").alias("fim")).first()["fim"]

window = compute_window(get_last_success_date(), SIM_DATE_OVERRIDE, DIAS_POR_EXECUCAO, SIM_START, SIM_END)

if window is None:
    dbutils.notebook.exit(f"Nada a fazer: todos os dias até {SIM_END} já foram gerados.")

SIM_DATE_START, SIM_DATE_END = window
print(f"Dias desta execução: {SIM_DATE_START} a {SIM_DATE_END} (último dia da simulação: {SIM_END})")

In [0]:
# O QUE FAZ: registra o início da execução em ops.simulator_runs com status RUNNING.
#   Se o run_id já existe (retentativa do job), a linha é reaberta em vez de duplicada.
#
# RECURSOS SQL:
# - MERGE INTO alvo USING origem ON condição: compara a origem com a tabela alvo;
#   WHEN MATCHED THEN UPDATE  -> a linha já existe: atualiza;
#   WHEN NOT MATCHED THEN INSERT -> não existe: insere.
# - (SELECT :run_id AS run_id, ...): origem de uma linha só, montada com os valores da execução.
# - :run_id, :inicio, :fim: parameter markers. O valor vai em args={...}, com o tipo preservado
#   (date vira DATE) e sem risco de SQL injection. Eles substituem valores, não nomes de tabela:
#   por isso o nome da tabela vem da constante RUNS_TABLE na f-string.
# - current_timestamp(): data/hora atual.
# MERGE pelo run_id: se o job repetir a task com o mesmo run_id, a linha é reaberta em vez de duplicada
spark.sql(
    f"""
    MERGE INTO {RUNS_TABLE} AS t
    USING (SELECT :run_id AS run_id, :inicio AS sim_date_start, :fim AS sim_date_end) AS s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET
        sim_date_start = s.sim_date_start,
        sim_date_end   = s.sim_date_end,
        status         = 'RUNNING',
        started_at     = current_timestamp(),
        finished_at    = NULL,
        files_written  = NULL,
        rows_written   = NULL,
        error_message  = NULL
    WHEN NOT MATCHED THEN INSERT (run_id, sim_date_start, sim_date_end, status, started_at)
        VALUES (s.run_id, s.sim_date_start, s.sim_date_end, 'RUNNING', current_timestamp())
    """,
    args={"run_id": RUN_ID, "inicio": SIM_DATE_START, "fim": SIM_DATE_END},
)

In [0]:
# O QUE FAZ: mostra a linha desta execução em simulator_runs (deve estar RUNNING).
# - spark.sql(texto, args={...}): SQL com parameter marker; display() mostra o resultado.
display(spark.sql(f"SELECT * FROM {RUNS_TABLE} WHERE run_id = :run_id", args={"run_id": RUN_ID}))

## Escrita determinística

Grava um arquivo JSON Lines por entidade e por dia simulado, no caminho definido no desenho (A3):

```
raw_data/<entidade>/sim_date=AAAA-MM-DD/<entidade>_AAAA-MM-DD_part-000.json
```

O Spark não permite escolher o nome do arquivo de saída (gera `part-00000-<uuid>.json`). Por isso a escrita acontece em duas fases:

1. **Preparação:** cada entidade é gravada em `raw_data/_staging/<run_id>/<entidade>/`, particionada por `sim_date`, ordenada pela chave e com uma partição por dia (um arquivo por dia).
2. **Publicação:** só depois que todas as entidades foram preparadas, os dias da janela são limpos no destino e cada arquivo é movido para o nome definitivo. A pasta de preparação é apagada ao final, com ou sem erro.

In [0]:
# O QUE FAZ: constantes da escrita: ordem das linhas de cada entidade (garante arquivo idêntico
#   a cada geração), entidades de carga inicial e a pasta de preparação desta execução.
#
# RECURSOS:
# - set {...}: conjunto sem ordem e sem repetição; "x in conjunto" é uma busca rápida.
# - RUN_STAGING_DIR: uma pasta por run_id, para duas execuções nunca misturarem arquivos.
# Ordem das linhas dentro de cada arquivo: garante conteúdo idêntico a cada geração (desenho A3)
ENTITY_SORT_KEYS = {
    "orders": ["order_id"],
    "order_status_events": ["order_id", "event_sequence"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "products": ["product_id"],
    # geolocation não tem chave e tem linhas repetidas: ordena por todas as colunas
    "geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state",
    ],
    "category_translation": ["product_category_name"],
}

# Entidades emitidas só no primeiro dia: fora dessa janela não precisam ser lidas
INITIAL_LOAD_ENTITIES = {"sellers", "products", "geolocation", "category_translation"}

RUN_STAGING_DIR = f"{STAGING_DIR}/{RUN_ID}"

In [0]:
# O QUE FAZ: declara funções puras de caminho e de datas usadas na escrita.
#
# FUNÇÕES DECLARADAS:
# - day_dir(entity, day): pasta definitiva do dia: raw_data/<entidade>/sim_date=AAAA-MM-DD.
# - final_file_name(entity, day): nome determinístico: <entidade>_<dia>_part-000.json.
# - days_between(start, end): lista com todos os dias de start até end (incluindo os dois).
#
# FUNÇÕES NATIVAS:
# - range(n): números de 0 a n-1. (end - start).days: número de dias entre as datas.
# - [... for i in range(...)]: list comprehension, monta a lista em uma linha.
def day_dir(entity: str, day: date | str) -> str:
    """Pasta definitiva de um dia: raw_data/<entidade>/sim_date=AAAA-MM-DD"""
    return f"{RAW_DIR}/{entity}/sim_date={day}"


def final_file_name(entity: str, day: date | str) -> str:
    """Nome definitivo do arquivo: só entidade, dia e número da parte (sem uuid, sem horário)."""
    return f"{entity}_{day}_part-000.json"


def days_between(start: date, end: date) -> list[date]:
    """Todos os dias do intervalo fechado [start, end]."""
    return [start + timedelta(days=i) for i in range((end - start).days + 1)]

In [0]:
# O QUE FAZ: declara a escrita em duas fases: prepara tudo na pasta _staging e, só se tudo deu
#   certo, limpa os dias no destino e publica os arquivos com o nome definitivo.
#
# FUNÇÕES DECLARADAS:
# - stage_entity(entity, df, start, end): grava os dias da janela em _staging/<run_id>/<entidade>/
#   em JSON, um arquivo por dia, com as linhas sempre na mesma ordem. Devolve o caminho.
# - publish_entity(entity, staging_path): para cada pasta sim_date=... da preparação, confere que
#   há exatamente 1 part-*.json e o move para o caminho definitivo. Devolve quantos publicou.
# - write_window(entities, start, end): orquestra as 3 fases (preparar, limpar, publicar) e
#   devolve (arquivos, linhas). Entidades de carga inicial só entram se a janela incluir SIM_START.
#
# FUNÇÕES NATIVAS:
# - Column.between(a, b): verdadeiro se a <= valor <= b (inclui os dois extremos).
# - df.repartition("sim_date"): redistribui as linhas (shuffle) para que todas as linhas do mesmo
#   dia fiquem na mesma partição. Assim cada dia sai em um único arquivo.
# - df.sortWithinPartitions(colunas): ordena dentro de cada partição, sem ordenação global.
# - df.write.mode("overwrite"): sobrescreve a pasta de destino (aqui, só a preparação da execução).
# - .partitionBy("sim_date"): cria uma subpasta sim_date=<valor> por dia e tira a coluna de
#   dentro do arquivo (o valor fica só no nome da pasta).
# - .option("ignoreNullFields", False): grava "campo": null em vez de omitir o campo.
# - .json(caminho): grava em JSON Lines (um objeto JSON por linha).
# - dbutils.fs.ls(caminho): lista a pasta; cada item tem name, path, size e modificationTime.
#   Nomes de pasta terminam com "/".
# - str.startswith / str.endswith: o texto começa/termina com...; removeprefix tira o prefixo;
#   rstrip("/") tira a barra do fim.
# - dbutils.fs.mkdirs(caminho): cria a pasta (e as pastas acima, se faltarem).
# - dbutils.fs.mv(origem, destino): move/renomeia o arquivo.
# - dbutils.fs.rm(caminho, recurse=True): apaga a pasta e tudo dentro dela.
# - spark.read.text(caminho).count(): lê os arquivos como texto (uma linha = uma linha do
#   DataFrame); em JSON Lines, linhas = registros.
# - raise RuntimeError(...): interrompe a execução com erro.
# - continue: pula para o próximo item do for.
def stage_entity(entity: str, df: DataFrame, start: date, end: date) -> str:
    """Grava os dias da janela na pasta de preparação: uma partição Spark por dia = um arquivo por dia."""
    path = f"{RUN_STAGING_DIR}/{entity}"
    (
        df
        .where(F.col("sim_date").between(start, end))
        .repartition("sim_date")
        .sortWithinPartitions(*ENTITY_SORT_KEYS[entity])
        .write
        .mode("overwrite")
        .partitionBy("sim_date")
        .option("ignoreNullFields", False)  # grava "campo": null em vez de omitir o campo
        .json(path)
    )
    return path


def publish_entity(entity: str, staging_path: str) -> int:
    """Move o único part-*.json de cada dia para o nome definitivo. Devolve o número de arquivos publicados."""
    published = 0
    for day_folder in dbutils.fs.ls(staging_path):
        if not day_folder.name.startswith("sim_date="):
            continue  # _SUCCESS e arquivos de controle do Spark
        day = day_folder.name.removeprefix("sim_date=").rstrip("/")

        parts = [f for f in dbutils.fs.ls(day_folder.path) if f.name.startswith("part-") and f.name.endswith(".json")]
        if len(parts) != 1:
            raise RuntimeError(f"{entity} {day}: esperado 1 arquivo na preparação, encontrados {len(parts)}")

        dbutils.fs.mkdirs(day_dir(entity, day))
        dbutils.fs.mv(parts[0].path, f"{day_dir(entity, day)}/{final_file_name(entity, day)}")
        published += 1
    return published


def write_window(entities: dict[str, DataFrame], start: date, end: date) -> tuple[int, int]:
    """Prepara todas as entidades, limpa os dias da janela no destino e publica. Devolve (arquivos, linhas)."""
    includes_initial_load = start <= SIM_START <= end
    selected = {
        entity: df
        for entity, df in entities.items()
        if entity not in INITIAL_LOAD_ENTITIES or includes_initial_load
    }

    # 1. Preparação: se falhar aqui, o destino continua intacto
    staged = {entity: stage_entity(entity, df, start, end) for entity, df in selected.items()}

    # 2. Limpeza do destino: reprocessar um dia substitui o que existia, sem sobras
    for entity in staged:
        for day in days_between(start, end):
            dbutils.fs.rm(day_dir(entity, day), recurse=True)

    # 3. Publicação (linhas contadas antes de mover: JSON Lines = 1 registro por linha)
    files_written, rows_written = 0, 0
    for entity, path in staged.items():
        rows_written += spark.read.text(path).count()
        files_written += publish_entity(entity, path)
    return files_written, rows_written

## Execução e finalização

Executa a escrita da janela e fecha o registro da execução em `olist.ops.simulator_runs`:

- **Sucesso:** `SUCCESS`, com a quantidade de arquivos e de linhas publicados. Só a partir daqui o progresso do simulador avança.
- **Erro:** `FAILED`, com a mensagem de erro; o erro é relançado para que a execução (e a task do job) termine como falha.
- Em qualquer caso, a pasta de preparação da execução é apagada.

In [0]:
# O QUE FAZ: declara a função que fecha a linha desta execução em simulator_runs.
#
# FUNÇÃO DECLARADA:
# - finish_run(status, files, rows, error_message): UPDATE da linha do RUN_ID com o status final
#   (SUCCESS ou FAILED), o horário de término, os totais e a mensagem de erro. Parâmetros com
#   padrão None: na falha, só status e mensagem são informados.
#
# RECURSOS:
# - UPDATE tabela SET coluna = valor WHERE condição: altera as linhas que atendem à condição.
# - Parameter markers: None do Python vira NULL no SQL.
# - "int | None": anotação de tipo; o valor pode ser inteiro ou None.
def finish_run(status: str, files: int | None = None, rows: int | None = None, error_message: str | None = None) -> None:
    """Fecha a linha da execução atual em simulator_runs."""
    spark.sql(
        f"""
        UPDATE {RUNS_TABLE}
        SET status        = :status,
            finished_at   = current_timestamp(),
            files_written = :files,
            rows_written  = :rows,
            error_message = :error_message
        WHERE run_id = :run_id
        """,
        args={"status": status, "files": files, "rows": rows, "error_message": error_message, "run_id": RUN_ID},
    )

In [0]:
# O QUE FAZ: executa a escrita da janela e registra o resultado.
#   Erro -> FAILED com a mensagem e o erro é relançado (a task do job falha).
#   Sempre -> a pasta de preparação desta execução é apagada.
#   Sem erro -> SUCCESS com os totais (fora do try: um erro aqui não pode marcar FAILED uma
#   execução que já publicou todos os arquivos).
#
# RECURSOS:
# - try / except Exception as error / finally: try executa; except roda só se houver erro;
#   finally roda sempre, com ou sem erro.
# - type(error).__name__: nome da classe do erro (ex.: "RuntimeError").
# - texto[:2000]: os primeiros 2.000 caracteres (limita o tamanho gravado na tabela).
# - raise (sozinho): relança o mesmo erro, com o traceback original.
try:
    files_written, rows_written = write_window(entities, SIM_DATE_START, SIM_DATE_END)
except Exception as error:
    finish_run("FAILED", error_message=f"{type(error).__name__}: {error}"[:2000])
    raise
finally:
    dbutils.fs.rm(RUN_STAGING_DIR, recurse=True)

finish_run("SUCCESS", files=files_written, rows=rows_written)

print(f"SUCCESS: {files_written} arquivos e {rows_written} linhas publicados ({SIM_DATE_START} a {SIM_DATE_END})")

In [0]:
# O QUE FAZ: validação: lista o arquivo publicado de cada entidade no primeiro dia da janela,
#   com o tamanho, e confirma que a pasta de preparação ficou vazia.
#
# RECURSOS:
# - try/except por entidade: dbutils.fs.ls lança erro se a pasta não existe (entidade sem
#   registro no dia), e a validação continua para as demais.
# - f.size: tamanho do arquivo em bytes.
# - "lista or 'vazia'": lista vazia conta como falso, então mostra 'vazia'.
for entity in ENTITY_SORT_KEYS:
    try:
        arquivos = [f"{f.name} ({f.size} bytes)" for f in dbutils.fs.ls(day_dir(entity, SIM_DATE_START))]
    except Exception:
        arquivos = ["(nenhum registro neste dia)"]
    print(f"{entity:22} {', '.join(arquivos)}")

print(f"\nPreparação: {[f.name for f in dbutils.fs.ls(STAGING_DIR)] or 'vazia'}")

In [0]:
# O QUE FAZ: mostra as 10 execuções mais recentes do simulador (status, janela, totais, erros).
# - ORDER BY started_at DESC LIMIT 10: ordena da mais nova para a mais antiga e pega 10.
display(spark.sql(f"SELECT * FROM {RUNS_TABLE} ORDER BY started_at DESC LIMIT 10"))